# Day 8: Web Scraping & Public APIs for Data Collection

## Calling a JSON API with requests
Most real data pipelines pull from an API, not a clean CSV - requests.get() plus .json() is the standard pattern.

In [ ]:
# pip install requests
import requests

response = requests.get("https://api.github.com/repos/pandas-dev/pandas")
data = response.json()
print(response.status_code)
print(data["full_name"], "-", data["stargazers_count"], "stars")

## Passing Parameters & Headers
Query parameters filter what an API returns; headers often carry an API key or content-type.

In [ ]:
import requests

params = {"q": "data science", "per_page": 3}
headers = {"Accept": "application/vnd.github+json"}

response = requests.get(
    "https://api.github.com/search/repositories",
    params=params,
    headers=headers,
)
for repo in response.json()["items"]:
    print(repo["full_name"], repo["stargazers_count"])

## Parsing HTML with BeautifulSoup
When there's no API, BeautifulSoup lets you pull structured data out of raw HTML by tag/class.

In [ ]:
# pip install beautifulsoup4
from bs4 import BeautifulSoup

html = """
<table>
  <tr><th>Name</th><th>Price</th></tr>
  <tr><td>Laptop</td><td>55000</td></tr>
  <tr><td>Mouse</td><td>799</td></tr>
</table>
"""
soup = BeautifulSoup(html, "html.parser")
rows = soup.find_all("tr")[1:]   # skip header row
for row in rows:
    name, price = [cell.text for cell in row.find_all("td")]
    print(name, "->", price)

## Turning Scraped/API Data into a DataFrame
The whole point of scraping is usually to land the result in pandas for cleaning and analysis.

In [ ]:
import pandas as pd

records = [
    {"name": "Laptop", "price": 55000},
    {"name": "Mouse", "price": 799},
]
df = pd.DataFrame(records)
print(df)

## Being a Good API Citizen
Real APIs rate-limit and can block you - respect limits, cache responses, and check terms of service / robots.txt before scraping a site.

In [ ]:
import time, requests

urls = ["https://api.github.com/repos/pandas-dev/pandas",
        "https://api.github.com/repos/numpy/numpy"]

for url in urls:
    r = requests.get(url)
    print(r.status_code, r.json().get("full_name"))
    time.sleep(1)   # be polite: don't hammer the server with rapid requests